In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd


In [3]:
torch.manual_seed(42)

In [4]:
if torch.backends.mps.is_available():
  device = torch.device('mps')
else:
  device = torch.device('cpu')

In [5]:
df_train = pd.read_csv('fashion-mnist_train.csv')
df_test = pd.read_csv('fashion-mnist_test.csv')

In [6]:
X_train = df_train.iloc[:,1:].values
y_train = df_train.iloc[:,0].values
X_test = df_test.iloc[:,1:].values
y_test = df_test.iloc[:,0].values

In [7]:
#scale the values
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
class Custom_dataset(Dataset):
  def __init__(self, features,labels):
     self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28) #no of rows unknown:-1
     self.labels = torch.tensor(labels,dtype=torch.long)
  def __len__(self):
     return len(self.features)

  def __getitem__(self,idx):
    return self.features[idx] , self.labels[idx]



In [9]:
# dataset and dataloader

train_dataset = Custom_dataset(X_train,y_train)
test_dataset = Custom_dataset(X_test,y_test)

train_dataloader = DataLoader(train_dataset,batch_size=32,shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset,batch_size=32,shuffle=False, pin_memory=True)




In [10]:
#Creating our CNN

class MyCNN(nn.Module):
  def __init__(self, input_features, num_class):
    super().__init__()
    self.features= nn.Sequential(

    #conv layer 1

    nn.Conv2d(input_features,32, kernel_size=3,padding='same'),
    nn.ReLU(),
    nn.BatchNorm2d(32),
    nn.MaxPool2d(kernel_size=2, stride =2),

    #conv layer 2

    nn.Conv2d(32, 64 , kernel_size=3,padding='same') ,
    nn.ReLU(),
    nn.BatchNorm2d(64),
    nn.MaxPool2d(kernel_size=2, stride =2)



    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear( 64*7*7 ,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(64,num_class)



    )
  def forward(self,x):
    feature = self.features(x)
    logits = self.classifier(feature)
    return logits


In [11]:
learning_rate = 0.01
epochs = 100

In [12]:
model = MyCNN(input_features=1,num_class=10)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate, weight_decay = 1e-4 )

In [13]:
#trainig loop

model.train()

for epoch in range(epochs):

  total_epoch_loss=0

  for batch_train, batch_labels in train_dataloader:

      batch_train = batch_train.to(device)
      batch_labels = batch_labels.to(device)

      #forward pass
      output = model(batch_train)

      loss = criterion(output,batch_labels)

      #backward pass
      optimizer.zero_grad()
      loss.backward()

      #update weights
      optimizer.step()

      total_epoch_loss+=loss.item()

  avg_loss = total_epoch_loss/len(train_dataloader)

  print(f'epoch: {epoch+1} , loss: {avg_loss}')




/opt/homebrew/Caskroom/miniforge/base/envs/ml_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


epoch: 1 , loss: 0.6679856689691543
epoch: 2 , loss: 0.4051165101846059
epoch: 3 , loss: 0.34335431053241094
epoch: 4 , loss: 0.30982149417002997
epoch: 5 , loss: 0.28359344573120276
epoch: 6 , loss: 0.2693565906743209
epoch: 7 , loss: 0.252596239978075
epoch: 8 , loss: 0.23920381113191447
epoch: 9 , loss: 0.22553066667119662
epoch: 10 , loss: 0.21422082086801528
epoch: 11 , loss: 0.20319227585395178
epoch: 12 , loss: 0.19238183379669985
epoch: 13 , loss: 0.18674809457858405
epoch: 14 , loss: 0.17996837816387415
epoch: 15 , loss: 0.16947261018157006
epoch: 16 , loss: 0.16537263034880162
epoch: 17 , loss: 0.1580388316536943
epoch: 18 , loss: 0.1514314643273751
epoch: 19 , loss: 0.1483290783688426
epoch: 20 , loss: 0.14246688484648865
epoch: 21 , loss: 0.13910576779171824
epoch: 22 , loss: 0.13300631974190474
epoch: 23 , loss: 0.1288429884875814
epoch: 24 , loss: 0.12351810276682178
epoch: 25 , loss: 0.11740526978547375
epoch: 26 , loss: 0.11412001234007378
epoch: 27 , loss: 0.1130104933

In [28]:
#evaluation

model.eval()
with torch.no_grad():

  total = 0

  correct = 0

  for batch_test, batch_labels in test_dataloader:

    batch_test = batch_test.to(device)
    batch_labels = batch_labels.to(device)

    output = model(batch_test)

    predictions = torch.argmax(output,dim=1)

    total= total+len(batch_labels)

    correct= correct+ (predictions==batch_labels).sum().item()

  accuracy= correct/total

  print(f'Test accuracy: {accuracy:.4f}')



/opt/homebrew/Caskroom/miniforge/base/envs/ml_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test accuracy: 0.9284


In [29]:
#training accuracy
total = 0

correct = 0
with torch.no_grad():

  
  for batch_test, batch_labels in train_dataloader:

    batch_test = batch_test.to(device)
    batch_labels = batch_labels.to(device)

    output = model(batch_test)

    predictions = torch.argmax(output,dim=1)

    total= total+ batch_labels.shape[0]

    correct= correct+ (predictions==batch_labels).sum().item()

accuracy = correct/total

print(f'Training accuracy: {accuracy:.4f}')

Training accuracy: 0.9995
